# Valutazione centralizzata: esperimento freezing su RegNetY-1.6GF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
RESULTS_DIR       = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
NUM_CLASSES = 8
SEED        = 1234
RECIPE_TAG  = 'acq_mild'

CLASS_NAMES = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

FAMILIES    = ['geometric', 'acquisition', 'background', 'resolution']
INTENSITIES = ['mild', 'moderate']

RUNS_TO_EVAL = {
    'full_ft (riferimento)': MODELS_DIR / 'regnety16gf_acq_mild.pth',
    'shallow_freeze':        MODELS_DIR / f'regnety16gf_{RECIPE_TAG}_shallow_freeze.pth',
    'mid_freeze':            MODELS_DIR / f'regnety16gf_{RECIPE_TAG}_mid_freeze.pth',
    'lp_ft':                 MODELS_DIR / f'regnety16gf_{RECIPE_TAG}_lp_ft.pth',
}

missing = [name for name, p in RUNS_TO_EVAL.items() if not p.exists()]
if missing:
    print("ATTENZIONE, checkpoint non ancora presenti:", missing)
else:
    print("Tutti i checkpoint sono presenti, si puo' procedere.")

Mounted at /content/drive
Tutti i checkpoint sono presenti, si puo' procedere.


In [ ]:
import io, random, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageEnhance, ImageFilter

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

if not DATASET_DIR.exists():
    print('Copio il dataset in locale...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Fatto in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente.')

df_all = pd.read_csv(SPLIT_CSV)
df_val = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f"Val: {len(df_val)}")

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

Device: cuda | Tesla T4
Copio il dataset in locale...
Fatto in 613s
Val: 3102


## Perturbazioni hard-val a 4 famiglie

In [ ]:
def _rng(idx, family, intensity):
    fam_id = {'geometric':1, 'acquisition':2, 'background':3, 'resolution':4}[family]
    int_id = {'mild':1, 'moderate':2}[intensity]
    return np.random.RandomState((SEED * 100003 + idx * 97 + fam_id * 13 + int_id) % (2**32))

def perturb_geometric(img, idx, intensity):
    r = _rng(idx, 'geometric', intensity)
    max_rot, min_scale = (8.0, 0.85) if intensity == 'mild' else (15.0, 0.60)
    W, H = img.size
    ang = r.uniform(-max_rot, max_rot)
    img = img.rotate(ang, resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
    scale = r.uniform(min_scale, 1.0)
    cw, ch = max(1, int(W * np.sqrt(scale))), max(1, int(H * np.sqrt(scale)))
    x0 = r.randint(0, max(1, W - cw + 1)); y0 = r.randint(0, max(1, H - ch + 1))
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((W, H), Image.BILINEAR)

def perturb_acquisition(img, idx, intensity):
    r = _rng(idx, 'acquisition', intensity)
    blur, q, jit, noise = (0.6, 70, 0.10, 4.0) if intensity == 'mild' else (1.2, 40, 0.20, 10.0)
    img = img.filter(ImageFilter.GaussianBlur(radius=blur * r.uniform(0.7, 1.3)))
    for Enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = Enh(img).enhance(1.0 + r.uniform(-jit, jit))
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=int(q)); buf.seek(0)
    img = Image.open(buf).convert('RGB')
    arr = np.asarray(img).astype(np.float32) + r.normal(0, noise, (img.size[1], img.size[0], 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def perturb_resolution(img, idx, intensity):
    r = _rng(idx, 'resolution', intensity)
    f = (0.5 if intensity == 'mild' else 0.34) * r.uniform(0.9, 1.1)
    W, H = img.size
    small = img.resize((max(1, int(W * f)), max(1, int(H * f))), Image.BILINEAR)
    return small.resize((W, H), Image.BILINEAR)

def perturb_background(img, idx, intensity):
    keep = 0.80 if intensity == 'mild' else 0.65
    W, H = img.size
    cw, ch = int(W * keep), int(H * keep)
    x0, y0 = (W - cw) // 2, (H - ch) // 2
    canvas = Image.new('RGB', (W, H), (255, 255, 255))
    canvas.paste(img.crop((x0, y0, x0 + cw, y0 + ch)), (x0, y0))
    return canvas

PERTURB = {'geometric': perturb_geometric, 'acquisition': perturb_acquisition,
           'background': perturb_background, 'resolution': perturb_resolution}

class ValDataset(Dataset):
    def __init__(self, df, family=None, intensity=None):
        self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
        self.family, self.intensity = family, intensity
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        img = Image.open(DATASET_DIR / fp).convert('RGB')
        if self.family is not None:
            img = PERTURB[self.family](img, i, self.intensity)
        return preprocess(img), lab

from sklearn.metrics import balanced_accuracy_score

@torch.no_grad()
def evaluate(model, df, family=None, intensity=None, bs=64):
    model.eval()
    dl = DataLoader(ValDataset(df, family, intensity), batch_size=bs, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    return balanced_accuracy_score(np.concatenate(ys), np.concatenate(ps))

## Valutazione delle 4 configurazioni

In [ ]:
def build_regnety_for_eval():
    m = models.regnet_y_1_6gf(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

rows = []
for run_name, wpath in RUNS_TO_EVAL.items():
    if not wpath.exists():
        print(f"[skip] {wpath.name} non trovato."); continue
    model = build_regnety_for_eval()
    model.load_state_dict(torch.load(wpath, map_location='cpu'))
    model.to(device)

    clean_bal = evaluate(model, df_val)
    fam_means, hard_vals = {}, []
    for fam in FAMILIES:
        vals = [evaluate(model, df_val, fam, inten) for inten in INTENSITIES]
        fam_means[fam] = float(np.mean(vals))
        hard_vals.extend(vals)
    all_fam_mean = float(np.mean(hard_vals))

    row = {'run': run_name, 'clean_val': round(clean_bal, 4),
           **{fam: round(v, 4) for fam, v in fam_means.items()},
           'hard_val_mean': round(all_fam_mean, 4)}
    rows.append(row)
    print(f"{run_name:24s} | clean={clean_bal:.4f} | hard-val mean={all_fam_mean:.4f}")
    del model; torch.cuda.empty_cache()

df_results = pd.DataFrame(rows)
out_csv = RESULTS_DIR / 'freeze_experiment_hardval.csv'
df_results.to_csv(out_csv, index=False)
print('\nSalvato:', out_csv)
df_results

full_ft (riferimento)    | clean=0.9799 | hard-val mean=0.9707
shallow_freeze           | clean=0.9822 | hard-val mean=0.9721
mid_freeze               | clean=0.9710 | hard-val mean=0.9501
lp_ft                    | clean=0.9750 | hard-val mean=0.9638

Salvato: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/results/freeze_experiment_hardval.csv


,run,clean_val,geometric,acquisition,background,resolution,hard_val_mean
0,full_ft (riferimento),0.9799,0.9751,0.9731,0.9603,0.9742,0.9707
1,shallow_freeze,0.9822,0.9788,0.9732,0.9603,0.9761,0.9721
2,mid_freeze,0.9710,0.9639,0.9477,0.9332,0.9556,0.9501
3,lp_ft,0.9750,0.9750,0.9677,0.9442,0.9684,0.9638
